## CELL 1 — Install Libraries

In [ ]:
# Install all required libraries
!pip install transformers -q        # HuggingFace: loads TrOCR pretrained model
!pip install easyocr -q             # EasyOCR: CRAFT+CRNN pretrained model (scene text)
!pip install torch torchvision -q   # PyTorch: deep learning framework
!pip install opencv-python-headless -q  # OpenCV: image analysis + preprocessing
!pip install Pillow -q              # PIL: image file handling
!pip install matplotlib -q          # matplotlib: visualization
!pip uninstall accelerate -y -q     # remove accelerate: causes meta tensor bug with TrOCR
!pip install gradio -q              # Gradio: web UI for ML models

print("All libraries installed!")
print("Please restart runtime now, then run cells top to bottom.")

## CELL 2 — Import Libraries

In [ ]:
# Core Python
import os                           # file path operations
import urllib.request               # download images from URL

# Image Processing
import cv2                          # OpenCV: image analysis and preprocessing
import numpy as np                  # NumPy: all images are NumPy arrays
from PIL import Image               # PIL: opens images for TrOCR input

# Visualization
import matplotlib.pyplot as plt     # plot images
import matplotlib.patches as patches  # draw bounding boxes

# OCR Models
import easyocr                      # EasyOCR: best for scene/natural text images
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
# TrOCRProcessor          : prepares image into model input tensor
# VisionEncoderDecoderModel : TrOCR model (ViT encoder + GPT2 decoder)

# PyTorch
import torch                        # PyTorch: tensor ops + GPU support

print(f"Imports successful!")
print(f"PyTorch   : {torch.__version__}")
print(f"GPU       : {torch.cuda.is_available()}")

## CELL 3 — Load Image
Two options: load from URL or upload from your PC.

In [ ]:
def download_image(url, save_path):
    """
    Downloads image from URL and saves to disk.

    Parameters:
        url       (str) : web address of image
        save_path (str) : local filename to save as

    Returns:
        save_path (str) : path of saved file
    """
    urllib.request.urlretrieve(url, save_path)  # download URL and save to disk
    print(f"Image saved: {save_path}")
    return save_path


def upload_image():
    """
    Opens file picker in Colab to upload image from PC.

    Parameters:
        None

    Returns:
        image_path (str) : filename of uploaded image
    """
    from google.colab import files       # Colab built-in upload helper
    uploaded   = files.upload()          # opens file picker dialog
    image_path = list(uploaded.keys())[0]  # get uploaded filename
    print(f"Uploaded: {image_path}")
    return image_path


def load_image(image_path):
    """
    Reads image from disk and returns as RGB NumPy array.

    Parameters:
        image_path (str) : path to image file

    Returns:
        img_rgb (numpy array) : image in RGB format shape (H x W x 3)

    Expected Output:
        Image displayed in notebook and shape printed.
    """
    img_bgr = cv2.imread(image_path)                         # read image (BGR format by default)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)       # convert BGR to RGB for matplotlib

    plt.figure(figsize=(8, 5))
    plt.imshow(img_rgb)
    plt.title("Input Image", fontsize=13)
    plt.axis('off')
    plt.show()

    print(f"Image shape: {img_rgb.shape}  (H x W x Channels)")
    return img_rgb


# CHOOSE ONE OPTION

# # OPTION A: Load from URL (default, no upload needed)
# IMAGE_URL  = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/af/Atomist_quote_from_Democritus.png/800px-Atomist_quote_from_Democritus.png"
# IMAGE_PATH = "input_image.png"
# download_image(IMAGE_URL, IMAGE_PATH)

# OPTION B: Upload from your PC
IMAGE_PATH = upload_image()
print(f"Using: {IMAGE_PATH}")

# Load and display the image
input_image = load_image(IMAGE_PATH)


# AGENT STEP 1 — Image Analyzer
### Measures 4 properties of the image to understand what type it is.
Each property has its own small function so you can see and explain each one independently.

In [ ]:
def measure_contrast(image):
    """
    Measures the contrast level of the image.
    Contrast = how much pixel brightness varies across the image.

    High contrast  means clean digital text (black text on white background).
    Low contrast   means faded, outlined, or scene text with background noise.

    Parameters:
        image (numpy array) : RGB image shape (H x W x 3)

    Returns:
        contrast (float) : standard deviation of pixel brightness (0 to 128)
                           greater than 50 means high contrast (clean document)
                           less than 50 means low contrast (scene or faded text)

    Expected Output:
        A single number representing contrast level.
    """
    gray     = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)  # convert to grayscale
    contrast = float(np.std(gray))  # std deviation of pixel values = contrast measure
                                     # high std means large brightness difference = high contrast
    print(f"   Contrast score : {contrast:.2f}  (>50 = high, <50 = low)")
    return contrast


def measure_noise(image):
    """
    Measures the noise level of the image.
    Noise = random pixel variations that are not part of actual content.

    High noise means photo or scene image (camera sensor noise, JPEG artifacts).
    Low noise  means clean digital or document image (computer generated, clean).

    How it works:
        Blur the image slightly then compare blurred vs original.
        The difference is noise because real content survives blur but noise gets removed.

    Parameters:
        image (numpy array) : RGB image shape (H x W x 3)

    Returns:
        noise (float) : mean pixel difference after blur (0 to 50)
                        greater than 8  means high noise (scene or photo)
                        8 or less means low noise (clean digital)

    Expected Output:
        A single number representing noise level.
    """
    gray    = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)   # convert to grayscale

    # GaussianBlur: slightly blur image to remove noise
    # (3,3) = 3x3 kernel, small blur to remove only noise not real content
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    # absdiff: absolute pixel-by-pixel difference between original and blurred
    # The difference shows how much noise existed in the original
    noise   = float(np.mean(cv2.absdiff(gray, blurred)))

    print(f"   Noise score    : {noise:.2f}  (>8 = high noise, <=8 = clean)")
    return noise


def measure_edge_density(image):
    """
    Measures how many edges (boundaries between light and dark) are in the image.

    High edge density means document or text image (many sharp text edges).
    Low edge density  means scene photo (smoother gradients, fewer sharp edges).

    How it works:
        Canny edge detector finds all sharp pixel boundaries.
        We count what percentage of pixels are edges.

    Parameters:
        image (numpy array) : RGB image shape (H x W x 3)

    Returns:
        edge_density (float) : percentage of pixels that are edges (0 to 1)
                               greater than 0.05 means text-heavy document
                               0.05 or less means scene or photo

    Expected Output:
        A decimal between 0 and 1 representing edge density.
    """
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)   # convert to grayscale

    # Canny: detects edges by finding sharp brightness changes
    # 50  = lower threshold: pixels below this are NOT edges
    # 150 = upper threshold: pixels above this ARE edges
    edges = cv2.Canny(gray, 50, 150)

    # Count edge pixels as fraction of total pixels
    edge_density = float(np.sum(edges > 0)) / edges.size

    print(f"   Edge density   : {edge_density:.4f}  (>0.05 = document, <=0.05 = scene)")
    return edge_density


def measure_color_variance(image):
    """
    Measures how colorful (varied) the image is.

    High color variance means scene or natural photo (many colors: sky, trees, signs).
    Low color variance  means document or text image (mostly black and white only).

    Parameters:
        image (numpy array) : RGB image shape (H x W x 3)

    Returns:
        color_var (float) : standard deviation of color channel differences (0 to 100)
                            greater than 20 means colorful scene image
                            20 or less means mostly black and white document

    Expected Output:
        A single number representing how colorful the image is.
    """
    r = image[:, :, 0].astype(float)   # Red channel pixels
    g = image[:, :, 1].astype(float)   # Green channel pixels
    b = image[:, :, 2].astype(float)   # Blue channel pixels

    # If R equals G equals B everywhere then image is grayscale or document (low variance)
    # If R, G, B are very different then image is colorful scene (high variance)
    rg_diff   = np.std(r - g)          # difference between Red and Green channels
    rb_diff   = np.std(r - b)          # difference between Red and Blue channels
    color_var = float((rg_diff + rb_diff) / 2)  # average color difference

    print(f"   Color variance : {color_var:.2f}  (>20 = colorful scene, <=20 = document)")
    return color_var


def analyze_image(image):
    """
    Runs all 4 analysis functions and returns measurements as a dictionary.
    This is the Agent's eyes. It observes the image before making any decisions.

    Parameters:
        image (numpy array) : RGB image shape (H x W x 3)

    Returns:
        features (dict) : dictionary with 4 measurement scores
                          'contrast'     = how sharp the brightness difference is
                          'noise'        = how much random pixel noise exists
                          'edge_density' = what fraction of pixels are edges
                          'color_var'    = how colorful the image is

    Expected Output:
        4 scores printed and returned as a dictionary.
    """
    print("=" * 50)
    print("STEP 1: ANALYZING IMAGE...")
    print("=" * 50)

    features = {
        'contrast'     : measure_contrast(image),      # brightness variation
        'noise'        : measure_noise(image),         # random pixel noise level
        'edge_density' : measure_edge_density(image),  # fraction of edge pixels
        'color_var'    : measure_color_variance(image) # color channel variation
    }

    print("=" * 50)
    print("Image analysis complete!")
    return features


# Run the analyzer on the loaded image
image_features = analyze_image(input_image)


#  AGENT STEP 2 — Image Classifier
### Uses the 4 measurements to classify the image into one of 3 types.

In [ ]:
def classify_image(features):
    """
    Classifies the image into one of 3 types using the measured features.
    This is the Agent's brain. It decides WHAT kind of image this is.

    Classification Rules:
        TYPE 1: 'scene'
            Natural or scene photo with text (street signs, shop boards, photos).
            Signal: high color variance (colorful) OR high noise.

        TYPE 2: 'handwritten'
            Handwritten text image (notes, forms, letters).
            Signal: low edge density and medium contrast.

        TYPE 3: 'document'
            Clean printed or digital text (PDFs, screenshots, typed docs).
            Signal: high contrast, low noise, high edge density.

    Parameters:
        features (dict) : output from analyze_image()
                          keys: 'contrast', 'noise', 'edge_density', 'color_var'

    Returns:
        image_type (str) : one of 'scene', 'handwritten', 'document'

    Expected Output:
        One of the 3 type labels printed with the reason for the decision.
    """
    print("=" * 50)
    print("STEP 2: CLASSIFYING IMAGE TYPE...")
    print("=" * 50)

    # Extract each score for easy reading
    contrast     = features['contrast']       # brightness variation score
    noise        = features['noise']          # noise level score
    edge_density = features['edge_density']   # edge fraction score
    color_var    = features['color_var']      # color variation score

    # RULE 1: Scene or natural image
    # Scene images are colorful (color_var > 20) OR very noisy (noise > 8)
    # Both indicate a real-world photo rather than a clean digital document
    if color_var > 20 or noise > 8:
        image_type = 'scene'
        reason = f"color_var={color_var:.1f} > 20 OR noise={noise:.1f} > 8 means natural scene photo"

    # RULE 2: Handwritten image
    # Handwriting has low edge density (soft pen strokes, not sharp machine-printed text)
    # AND medium contrast (not as sharp as printed, not as noisy as scene)
    elif edge_density < 0.03 and contrast < 60:
        image_type = 'handwritten'
        reason = f"edge_density={edge_density:.4f} < 0.03 AND contrast={contrast:.1f} < 60 means handwriting"

    # RULE 3: Document or printed text (default for everything else)
    # High contrast, low noise, high edge density = clean digital text
    else:
        image_type = 'document'
        reason = f"high contrast={contrast:.1f} and low noise={noise:.1f} means clean document"

    print(f"   Reason     : {reason}")
    print(f"   Image type : '{image_type}'")
    print("=" * 50)
    return image_type


# Run the classifier using the features from Step 1
image_type = classify_image(image_features)


# AGENT STEP 3 — Model Selector
### Uses the image type to pick the best pretrained OCR model.

In [ ]:
def select_model(image_type):
    """
    Selects the best pretrained OCR model based on the image type.
    This is the Agent's decision step. It picks the right tool for the job.

    Selection Table:
        'scene'       -> EasyOCR           (CRAFT+CRNN, trained on real-world scene photos)
        'handwritten' -> trocr-handwritten (TrOCR ViT+GPT2, trained on IAM handwriting)
        'document'    -> trocr-printed     (TrOCR ViT+GPT2, trained on printed documents)

    Parameters:
        image_type (str) : one of 'scene', 'handwritten', 'document'

    Returns:
        model_name (str) : identifier for the selected model
                           'easyocr'           = use EasyOCR
                           'trocr-handwritten' = use TrOCR handwritten variant
                           'trocr-printed'     = use TrOCR printed variant

    Expected Output:
        Model name printed with explanation of why it was chosen.
    """
    print("=" * 50)
    print("STEP 3: SELECTING BEST MODEL...")
    print("=" * 50)

    # Lookup dictionary: image type maps to best model name
    # Using a dictionary is cleaner than a long if/elif chain for simple mapping
    model_map = {
        'scene'       : 'easyocr',           # EasyOCR specializes in scene text
        'handwritten' : 'trocr-handwritten', # TrOCR handwritten variant
        'document'    : 'trocr-printed'      # TrOCR printed variant
    }

    # Explanation of why each model was chosen (for display)
    reason_map = {
        'easyocr'           : 'EasyOCR CRAFT+CRNN pretrained on real-world scene text photos',
        'trocr-handwritten' : 'TrOCR ViT+GPT2 finetuned on IAM handwriting dataset',
        'trocr-printed'     : 'TrOCR ViT+GPT2 finetuned on printed document dataset'
    }

    model_name = model_map[image_type]      # look up model for this image type
    reason     = reason_map[model_name]     # look up reason for display

    print(f"   Image type   : '{image_type}'")
    print(f"   Model chosen : '{model_name}'")
    print(f"   Reason       : {reason}")
    print("=" * 50)
    return model_name


# Run the model selector using the image type from Step 2
selected_model = select_model(image_type)


# AGENT STEP 4 — Preprocessor
### Applies the RIGHT preprocessing for the selected model automatically.

In [ ]:
def preprocess_for_easyocr(image):
    """
    Prepares image for EasyOCR (best for scene and natural text images).
    EasyOCR needs a color image at moderate size with minimal processing.
    Reason: CRAFT detector was trained on natural color images.

    Parameters:
        image (numpy array) : original RGB image shape (H x W x 3)

    Returns:
        processed (numpy array) : resized color image ready for EasyOCR

    Expected Output:
        Color image resized to max 1200px width.
    """
    print("   Preprocessing for EasyOCR (scene text)...")

    h, w  = image.shape[:2]             # get original height and width
    scale = min(1200 / w, 1000 / h)     # scale to fit within 1200x1000 box
                                         # min() ensures neither dimension exceeds limit

    if scale < 1.0:                      # only resize if image is larger than limit
        new_w     = int(w * scale)
        new_h     = int(h * scale)
        # INTER_AREA: best interpolation method for downscaling (reduces aliasing)
        processed = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    else:
        processed = image.copy()         # keep original size if already small enough

    print(f"      Original: {w}x{h}  Processed: {processed.shape[1]}x{processed.shape[0]}")
    return processed


def preprocess_for_trocr(image):
    """
    Prepares image for TrOCR (best for printed documents and handwriting).
    TrOCR needs the image upscaled for better character resolution per patch.
    Reason: ViT encoder reads image as 16x16 pixel patches and needs enough pixels per patch.

    Parameters:
        image (numpy array) : original RGB image shape (H x W x 3)

    Returns:
        processed (numpy array) : upscaled color image ready for TrOCR

    Expected Output:
        Color image upscaled 3x for better character recognition.
    """
    print("   Preprocessing for TrOCR (printed or handwritten)...")

    h, w = image.shape[:2]

    # Upscale 3x: gives TrOCR more pixels per character to read
    # INTER_CUBIC: best interpolation for upscaling (smooth, no blocky pixels)
    processed = cv2.resize(
        image,
        (w * 3, h * 3),                 # triple width and height
        interpolation=cv2.INTER_CUBIC   # smooth upscaling algorithm
    )

    print(f"      Original: {w}x{h}  Upscaled: {processed.shape[1]}x{processed.shape[0]} (3x)")
    return processed


def preprocess_image(image, model_name):
    """
    Master preprocessor: automatically routes to the correct preprocessing function
    based on which model was selected in Step 3.
    This is the Agent's preparation step before running OCR.

    Parameters:
        image      (numpy array) : original RGB image shape (H x W x 3)
        model_name (str)         : model selected in Step 3
                                   'easyocr'           = scene preprocessing
                                   'trocr-printed'     = TrOCR preprocessing
                                   'trocr-handwritten' = TrOCR preprocessing

    Returns:
        processed (numpy array) : correctly preprocessed image

    Expected Output:
        Before and after images displayed side by side.
    """
    print("=" * 50)
    print("STEP 4: PREPROCESSING IMAGE...")
    print("=" * 50)

    if model_name == 'easyocr':
        processed = preprocess_for_easyocr(image)        # scene: resize only

    elif model_name in ('trocr-printed', 'trocr-handwritten'):
        processed = preprocess_for_trocr(image)          # document or handwritten: upscale 3x

    else:
        processed = image.copy()                          # unknown model: no preprocessing
        print("   Unknown model, no preprocessing applied")

    # Display before and after side by side
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(image)
    axes[0].set_title("Before Preprocessing", fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(processed)
    axes[1].set_title(f"After Preprocessing (for {model_name})", fontsize=12, color='green')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    print("Preprocessing done!")
    print("=" * 50)
    return processed


# Run preprocessor using selected model name from Step 3
processed_image = preprocess_image(input_image, selected_model)


# AGENT STEP 5 — Model Loader
### Loads ONLY the selected model. Other models are not loaded (saves memory).

In [ ]:
def load_easyocr():
    """
    Loads EasyOCR pretrained model (CRAFT text detector + CRNN text recognizer).
    Best for: scene text, street signs, natural photos.

    Parameters:
        None

    Returns:
        reader (easyocr.Reader) : loaded EasyOCR model ready to use

    Expected Output:
        EasyOCR model loaded with confirmation message.
    """
    print("   Loading EasyOCR (CRAFT + CRNN)...")

    # easyocr.Reader: downloads and loads CRAFT and CRNN pretrained weights
    # ['en']                         = English language model
    # gpu=torch.cuda.is_available()  = use GPU if available, else use CPU
    reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

    print("   EasyOCR loaded!")
    return reader


def load_trocr(model_name):
    """
    Loads Microsoft TrOCR pretrained model from HuggingFace.
    Best for: printed documents OR handwritten text depending on variant.

    Parameters:
        model_name (str) : which TrOCR variant to load
                           'trocr-printed'     = microsoft/trocr-base-printed
                           'trocr-handwritten' = microsoft/trocr-base-handwritten

    Returns:
        processor (TrOCRProcessor)           : converts image to model input tensor
        model     (VisionEncoderDecoderModel): ViT encoder + GPT2 decoder model

    Expected Output:
        TrOCR model loaded and set to eval mode.
    """
    # Map our short model name to the actual HuggingFace model path
    hf_model_map = {
        'trocr-printed'     : 'microsoft/trocr-base-printed',      # trained on printed docs
        'trocr-handwritten' : 'microsoft/trocr-base-handwritten'   # trained on handwriting
    }

    hf_path = hf_model_map[model_name]    # get the actual HuggingFace model identifier
    print(f"   Loading TrOCR: {hf_path}...")

    # TrOCRProcessor: handles image resizing and normalization before model input
    processor = TrOCRProcessor.from_pretrained(hf_path)

    # VisionEncoderDecoderModel: the full TrOCR model (ViT encoder + GPT2 decoder)
    model = VisionEncoderDecoderModel.from_pretrained(hf_path)

    # eval mode: disables dropout layers, no gradient updates needed for inference
    model.eval()

    print(f"   TrOCR loaded! ({hf_path})")
    return processor, model


def load_model(model_name):
    """
    Master model loader: loads ONLY the model that was selected in Step 3.
    Agent loads only what it needs to save memory and loading time.

    Parameters:
        model_name (str) : output from select_model()
                           one of: 'easyocr', 'trocr-printed', 'trocr-handwritten'

    Returns:
        loaded_model (dict) : dictionary containing model object(s) and model name
                              For EasyOCR:   keys are 'name' and 'reader'
                              For TrOCR:     keys are 'name', 'processor', 'model'

    Expected Output:
        Only the selected model is loaded. All other models are skipped.
    """
    print("=" * 50)
    print("STEP 5: LOADING SELECTED MODEL...")
    print("=" * 50)
    print(f"   Loading '{model_name}' only. Other models are skipped to save memory.")

    if model_name == 'easyocr':
        reader = load_easyocr()
        loaded_model = {
            'name'   : 'easyocr',
            'reader' : reader           # EasyOCR uses a single reader object
        }

    elif model_name in ('trocr-printed', 'trocr-handwritten'):
        processor, model = load_trocr(model_name)
        loaded_model = {
            'name'      : model_name,
            'processor' : processor,    # TrOCR uses processor to prepare image input
            'model'     : model         # TrOCR model for generating text
        }

    print("=" * 50)
    print(f"Model '{model_name}' is ready!")
    return loaded_model


# Load only the selected model
loaded_model = load_model(selected_model)


# AGENT STEP 6 — OCR Runner
### Runs the selected model on the preprocessed image and extracts text.

In [ ]:
def run_easyocr(reader, image):
    """
    Runs EasyOCR on the image to extract text.

    Parameters:
        reader (easyocr.Reader) : loaded EasyOCR model
        image  (numpy array)    : preprocessed color image

    Returns:
        results (list) : list of tuples ( [bbox], 'text', confidence )

    Expected Output:
        List of detected text regions with bounding boxes and confidence scores.
    """
    # reader.readtext: runs CRAFT to detect text boxes then CRNN to read text
    # text_threshold=0.4  : lower than default 0.7, catches more text regions
    # low_text=0.3        : lower than default 0.4, catches weak character boundaries
    # link_threshold=0.3  : lower than default 0.4, links characters more freely
    # contrast_ths=0.05   : lower than default 0.1, allows low contrast text
    # adjust_contrast=0.7 : higher than default 0.5, boosts contrast internally
    # detail=1            : return full output including bbox, text and confidence
    # paragraph=False     : keep each detected text box separate, not merged
    raw = reader.readtext(
        image,
        text_threshold  = 0.4,
        low_text        = 0.3,
        link_threshold  = 0.3,
        contrast_ths    = 0.05,
        adjust_contrast = 0.7,
        detail          = 1,
        paragraph       = False
    )

    # EasyOCR already returns (bbox, text, confidence) format so use directly
    results = [(bbox, text, conf) for bbox, text, conf in raw]
    return results


def run_trocr(processor, model, image):
    """
    Runs TrOCR on the image to extract text line by line.
    TrOCR processes one line at a time so we must split image into lines first.

    Parameters:
        processor (TrOCRProcessor)           : converts image crops to tensors
        model     (VisionEncoderDecoderModel): TrOCR ViT+GPT2 model
        image     (numpy array)              : preprocessed color image

    Returns:
        results (list) : list of tuples ( [bbox], 'text', confidence=1.0 )

    Expected Output:
        List of detected text lines. TrOCR does not return confidence scores.
    """

    def detect_lines(image):
        """
        Finds bounding boxes for each text line using OpenCV contours.
        TrOCR reads one line at a time so we must locate each line first.

        Parameters:
            image (numpy array) : color image

        Returns:
            regions (list) : list of (x, y, w, h) tuples sorted top to bottom
        """
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)   # convert to grayscale

        # THRESH_BINARY_INV makes text white (255) and background black (0)
        # THRESH_OTSU automatically calculates the best threshold value
        _, binary = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # Dilate horizontally to merge all characters on the same line into one blob
        # (80, 2) kernel: 80px wide merges chars across a line, 2px tall keeps lines separate
        kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (80, 2))
        dilated = cv2.dilate(binary, kernel, iterations=2)  # expand blobs horizontally

        # Find contours where each contour is one connected blob = one text line
        contours, _ = cv2.findContours(
            dilated,
            cv2.RETR_EXTERNAL,       # only find outermost contours
            cv2.CHAIN_APPROX_SIMPLE  # compress contour points to save memory
        )

        # Get bounding box of each blob and filter out tiny noise blobs
        regions = [
            cv2.boundingRect(c) for c in contours
            if cv2.boundingRect(c)[2] > 50    # width must be greater than 50px
            and cv2.boundingRect(c)[3] > 10   # height must be greater than 10px
        ]

        # Sort regions top to bottom by y coordinate (natural reading order)
        return sorted(regions, key=lambda r: r[1])

    regions      = detect_lines(image)         # get all line bounding boxes
    h_img, w_img = image.shape[:2]             # image height and width for boundary clamping
    results      = []

    for x, y, w, h in regions:
        # Add 5px padding around each crop so letter edges are not cut off
        x1 = max(0,     x - 5)        # left edge with padding, clamped to image boundary
        y1 = max(0,     y - 5)        # top edge with padding, clamped to image boundary
        x2 = min(w_img, x + w + 5)    # right edge with padding, clamped to image boundary
        y2 = min(h_img, y + h + 5)    # bottom edge with padding, clamped to image boundary

        crop     = image[y1:y2, x1:x2]   # crop just this one text line from full image
        pil_crop = Image.fromarray(crop)   # convert numpy array to PIL (TrOCR needs PIL format)

        # processor resizes crop to 384x384 and normalizes pixel values to model input format
        # return_tensors='pt' means return PyTorch tensors not numpy arrays
        pixel_values = processor(
            images=pil_crop,
            return_tensors='pt'
        ).pixel_values                     # pixel_values is the tensor ready for the model

        # model.generate: ViT encoder reads image then GPT2 decoder generates text tokens
        # max_new_tokens=64: maximum number of characters to generate per line
        # torch.no_grad() skips gradient calculation since we only need inference not training
        with torch.no_grad():
            generated_ids = model.generate(pixel_values, max_new_tokens=64)

        # Decode token IDs back into a readable text string
        # skip_special_tokens=True removes BOS and EOS control tokens from output
        text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0]                               # [0] gets the first and only item from batch

        bbox = [[x1,y1],[x2,y1],[x2,y2],[x1,y2]]  # 4-corner bounding box format
        results.append((bbox, text, 1.0))           # confidence is 1.0 because TrOCR has no score

    return results


def run_ocr(loaded_model, image):
    """
    Master OCR runner: routes to the correct OCR function based on which model was loaded.

    Parameters:
        loaded_model (dict)       : output from load_model()
                                    contains 'name' plus model objects
        image        (numpy array): preprocessed image ready for OCR

    Returns:
        results (list) : list of tuples ( [bbox], 'text', confidence )

    Expected Output:
        Table of detected text lines printed with confidence scores.
    """
    print("=" * 50)
    print("STEP 6: RUNNING OCR...")
    print("=" * 50)

    name = loaded_model['name']    # which model is loaded
    print(f"   Using model: '{name}'")

    if name == 'easyocr':
        results = run_easyocr(loaded_model['reader'], image)

    elif name in ('trocr-printed', 'trocr-handwritten'):
        results = run_trocr(loaded_model['processor'], loaded_model['model'], image)

    else:
        print("   Unknown model name. Cannot run OCR.")
        return []

    # Print results table
    print(f"\n   Found {len(results)} text regions:")
    print(f"   {'-'*48}")
    print(f"   {'#':<4} {'Text':<35} {'Conf':>6}")
    print(f"   {'-'*48}")
    for i, (_, text, conf) in enumerate(results, 1):
        print(f"   {i:<4} {text:<35} {conf:>5.0%}")
    print(f"   {'-'*48}")
    print("=" * 50)
    return results


# Run OCR using the loaded model and preprocessed image
ocr_results = run_ocr(loaded_model, processed_image)


# AGENT STEP 7 — Visualize and Final Output
### Draw bounding boxes and print the clean final extracted text.

In [ ]:
def visualize_results(image, results, model_name):
    """
    Draws colored bounding boxes on the image for each detected text region.

    Parameters:
        image      (numpy array) : preprocessed image used for OCR
        results    (list)        : output from run_ocr()
        model_name (str)         : which model was used (shown in the title)

    Returns:
        None (displays annotated image in notebook)

    Expected Output:
        Image with colored boxes around all detected text regions.
        Green = high confidence, Yellow = medium, Red = low confidence.
    """
    fig, ax = plt.subplots(1, figsize=(14, 9))
    ax.imshow(image)

    for bbox, text, conf in results:
        pts   = np.array(bbox)               # 4 corner points of detected text box
        x_min = pts[:, 0].min()              # leftmost x coordinate
        y_min = pts[:, 1].min()              # topmost y coordinate
        box_w = pts[:, 0].max() - x_min      # box width in pixels
        box_h = pts[:, 1].max() - y_min      # box height in pixels

        # Color by confidence level
        # lime = high confidence above 70 percent
        # yellow = medium confidence between 40 and 70 percent
        # red = low confidence below 40 percent
        color = 'lime' if conf > 0.7 else ('yellow' if conf > 0.4 else 'red')

        # Draw rectangle around detected text
        ax.add_patch(patches.Rectangle(
            (x_min, y_min), box_w, box_h,
            linewidth=2, edgecolor=color, facecolor='none'  # facecolor='none' = transparent fill
        ))

        # Add text label just above the bounding box
        ax.text(
            x_min, y_min - 5,
            f"{text} ({conf:.0%})",
            color='white', fontsize=8,
            bbox=dict(facecolor=color, alpha=0.7, pad=1)
        )

    ax.set_title(f"OCR Results | Model: {model_name} | {len(results)} regions detected", fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print("Green = high confidence >70%")
    print("Yellow = medium confidence 40-70%")
    print("Red = low confidence <40%")


def get_final_text(results):
    """
    Combines all detected text into one clean readable string.
    Also fixes ALL CAPS output which trocr-base-printed produces.

    Parameters:
        results (list) : output from run_ocr()

    Returns:
        final_text (str) : all extracted text joined as one string

    Expected Output:
        Clean readable text with correct sentence case.
    """
    lines = [text for _, text, _ in results]   # collect all detected text strings
    raw   = '\n'.join(lines)                   # join with newlines

    # Fix ALL CAPS: trocr-base-printed outputs uppercase due to its training data
    # Convert each line to sentence case: first letter uppercase, rest lowercase
    fixed_lines = []
    for line in raw.split('\n'):
        line = line.strip()                    # remove leading and trailing spaces
        if line:                               # skip empty lines
            fixed_lines.append(line[0].upper() + line[1:].lower())
    final_text = '\n'.join(fixed_lines)

    print("=" * 50)
    print("FINAL EXTRACTED TEXT")
    print("=" * 50)
    print(final_text)
    print("=" * 50)
    print(f"   Characters : {len(final_text)}")
    print(f"   Words      : {len(final_text.split())}")
    return final_text


# Visualize the OCR results on the image
visualize_results(processed_image, ocr_results, selected_model)

# Print the final clean extracted text
final_text = get_final_text(ocr_results)

---
# FULL AGENT — One Function That Runs Everything
### Give it any image path and it handles all 6 steps automatically.
This is what you demo in the interview.

In [ ]:
def run_agent(image_path):
    """
    THE MASTER AGENT FUNCTION.
    Give it any image path and it automatically runs all steps:

        Step 1: Load image from disk
        Step 2: Analyze image (contrast, noise, edges, color variance)
        Step 3: Classify image type (scene, document, handwritten)
        Step 4: Select best pretrained model for that image type
        Step 5: Apply correct preprocessing for the selected model
        Step 6: Load only the selected model (saves memory)
        Step 7: Run OCR and extract text
        Step 8: Visualize results and return clean text

    Parameters:
        image_path (str) : path to any image file on disk

    Returns:
        final_text (str) : all extracted text from the image

    Expected Output:
        Full pipeline output with all steps printed and image annotated.
    """
    print("\n" + "#" * 50)
    print("#  OCR AI AGENT STARTING")
    print("#" * 50)

    image      = load_image(image_path)              # Step 1: load image as RGB array
    features   = analyze_image(image)                # Step 2: measure 4 image properties
    img_type   = classify_image(features)            # Step 3: classify as scene/document/handwritten
    model_name = select_model(img_type)              # Step 4: select best pretrained model
    processed  = preprocess_image(image, model_name) # Step 5: apply correct preprocessing
    model_obj  = load_model(model_name)              # Step 6: load selected model only
    results    = run_ocr(model_obj, processed)       # Step 7: run OCR and get detections
    visualize_results(processed, results, model_name)# Step 8a: draw bounding boxes
    text       = get_final_text(results)             # Step 8b: get clean final text

    print("\n" + "#" * 50)
    print("#  AGENT COMPLETE")
    print(f"#  Image type : {img_type}")
    print(f"#  Model used : {model_name}")
    print(f"#  Words found: {len(text.split())}")
    print("#" * 50)
    return text


# Run the full agent on your image
# To use your own image: change IMAGE_PATH to your uploaded file path
extracted_text = run_agent(IMAGE_PATH)

## Gradio App for UI

In [ ]:
import gradio as gr          # gradio: web UI library for ML models

def agent_for_gradio(uploaded_image):
    """
    Wrapper function that connects Gradio UI to our run_agent() pipeline.
    Gradio passes image as a numpy array directly so no file path needed.

    Parameters:
        uploaded_image (numpy array) : image uploaded by user through Gradio UI
                                       Gradio automatically converts uploaded file
                                       to numpy array in RGB format (H x W x 3)

    Returns:
        image_type  (str) : detected image type (scene / document / handwritten)
        model_used  (str) : name of model selected by agent
        final_text  (str) : all extracted text from the image

    Expected Output:
        3 output boxes filled in the Gradio UI automatically.
    """
    # Step 1: Analyze the uploaded image
    features   = analyze_image(uploaded_image)         # get 4 measurement scores

    # Step 2: Classify image type
    img_type   = classify_image(features)              # scene / document / handwritten

    # Step 3: Select best model
    model_name = select_model(img_type)                # easyocr / trocr-printed / etc.

    # Step 4: Preprocess image correctly for selected model
    processed  = preprocess_image(uploaded_image, model_name)  # resize or upscale

    # Step 5: Load selected model only
    model_obj  = load_model(model_name)                # load weights into memory

    # Step 6: Run OCR
    results    = run_ocr(model_obj, processed)         # extract text detections

    # Step 7: Get clean final text
    text       = get_final_text(results)               # join + fix case

    # Return 3 values — each goes to one output box in the Gradio UI
    return img_type, model_name, text

## Gradio Interface

In [ ]:
demo = gr.Interface(

    fn      = agent_for_gradio,    # function to call when user clicks Submit
                                    # receives image input, returns 3 outputs

    inputs  = gr.Image(            # input widget: image upload box
        type  = "numpy",           # type='numpy': gives us numpy array directly
                                    # other options: 'pil' (PIL Image), 'filepath' (string)
        label = "Upload Image"     # label shown above the upload box in UI
    ),

    outputs = [
        gr.Textbox(label="Image Type Detected"),   # shows: scene/document/handwritten
        gr.Textbox(label="Model Selected"),        # shows: easyocr/trocr-printed/etc.
        gr.Textbox(
            label     = "Extracted Text",  # label shown above the text box
            lines     = 10,                # lines=10: sets default visible height to 10 lines
                                            # increase this number to make box taller
            max_lines = 50,                # max_lines=50: box can grow up to 50 lines tall
                                           # user can resize by dragging bottom-right corner
            show_copy_button = True        # adds a copy button so user can copy text easily
        )
    ],

    title       = "OCR AI Agent",                  # title shown at top of UI
    description = "Upload any image containing text. The agent will automatically "
                  "analyze the image, select the best pretrained OCR model, "
                  "and extract all text.",          # subtitle shown below title

    examples    = [[IMAGE_PATH]],                  # shows sample image button in UI
                                                    # IMAGE_PATH = our sample image loaded in Cell 3

    allow_flagging = "never"                       # 'never': hides the Flag button from UI
                                                    # other options: 'auto', 'manual'
)

# Launch the app

In [ ]:
demo.launch(
    share   = True,    # share=True: generates a public URL like https://xxx.gradio.live
                        # anyone with the link can use your app for 72 hours
                        # share=False: only accessible inside Colab (localhost)
    debug   = False    # debug=False: hides internal error tracebacks from user
                        # debug=True: shows full error details (use during development)
)
